# **Data Mining Assignment: Apache Spark Data Lake Preprocessing**

#### Rupsa Roy
#### MSD25007

In [54]:
!pip install pyspark

In [55]:
import os
import shutil
import urllib.request

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum, when, trim, initcap, coalesce, expr, date_format
from pyspark.sql.types import StringType

spark = SparkSession.builder.appName('Dataframe').getOrCreate()

In [56]:
os.makedirs("data_lake/bronze/", exist_ok=True)
os.makedirs("data_lake/silver/", exist_ok=True)

url = "https://raw.githubusercontent.com/necromancersanju-netizen/data-mining-data-lake-assignment/main/dataset/sales.csv"
urllib.request.urlretrieve(url, "data_lake/bronze/sales.csv")

('data_lake/bronze/sales.csv', <http.client.HTTPMessage at 0x78c9662ca8d0>)

In [57]:
df = spark.read.csv("data_lake/bronze/sales.csv", header=True, inferSchema=True)
df.show(5)

+--------+-----------+--------------+-------------+--------+--------+----------+----------------+--------------+----------+--------------+----------+------------+
|order_id|customer_id| customer_name|      product|category|quantity|unit_price|discount_percent|payment_method|      city|         state|order_date|order_status|
+--------+-----------+--------------+-------------+--------+--------+----------+----------------+--------------+----------+--------------+----------+------------+
|  100001|      C0256|Keerthi Mishra|        Saree|Clothing|       2|      6710|              20|   Credit Card|    Mumbai|   Maharashtra|2026-01-01|   Delivered|
|  100002|      C0042|   Arjun Singh|      T-Shirt|Clothing|       6|       680|               5|    Debit Card|   Chennai|    Tamil Nadu|2025-08-03|   Delivered|
|  100003|      C0348|   Keerthi Rao|       Hoodie|Clothing|       6|      3080|              12|           UPI|      Pune|   Maharashtra|2025-05-06|     Shipped|
|  100004|      C0051|

## Displaying the schema


In [58]:
df.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: integer (nullable = true)
 |-- discount_percent: integer (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- order_status: string (nullable = true)



## Total number of records before preprocessing

In [59]:
print("Rows:", df.count())

Rows: 1000


## Duplicate records

In [60]:
print("Unique Records:", df.dropDuplicates().count())
print("Duplicate Records:", df.count() - df.dropDuplicates().count())

Unique Records: 990
Duplicate Records: 10


## The number of columns and their names

In [61]:
print("Columns:", len(df.columns))
print("Column Names:", df.columns)

Columns: 13
Column Names: ['order_id', 'customer_id', 'customer_name', 'product', 'category', 'quantity', 'unit_price', 'discount_percent', 'payment_method', 'city', 'state', 'order_date', 'order_status']


## Missing values

In [62]:
missing_values = df.select([
    sum((col(c).isNull() | (trim(col(c).cast("string")) == "")).cast("int")).alias(c)
    for c in df.columns
])
missing_values.show()

+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+
|order_id|customer_id|customer_name|product|category|quantity|unit_price|discount_percent|payment_method|city|state|order_date|order_status|
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+
|       0|          0|           10|      0|       0|       0|         0|               0|             8|   8|    0|         0|           0|
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+



## Distinct text values

In [63]:
for c in df.columns:
    if isinstance(df.schema[c].dataType, StringType):
        print("Column:", c)
        df.select(c).distinct().show(50, False)

Column: customer_id
+-----------+
|customer_id|
+-----------+
|C0114      |
|C0141      |
|C0243      |
|C0034      |
|C0154      |
|C0347      |
|C0130      |
|C0338      |
|C0167      |
|C0179      |
|C0195      |
|C0150      |
|C0033      |
|C0107      |
|C0016      |
|C0131      |
|C0191      |
|C0261      |
|C0057      |
|C0280      |
|C0046      |
|C0284      |
|C0102      |
|C0061      |
|C0337      |
|C0028      |
|C0058      |
|C0050      |
|C0139      |
|C0109      |
|C0325      |
|C0006      |
|C0185      |
|C0049      |
|C0215      |
|C0247      |
|C0282      |
|C0300      |
|C0170      |
|C0305      |
|C0051      |
|C0163      |
|C0346      |
|C0147      |
|C0285      |
|C0200      |
|C0076      |
|C0084      |
|C0314      |
|C0224      |
+-----------+
only showing top 50 rows
Column: customer_name
+------------------+
|customer_name     |
+------------------+
|Pranav Jain       |
|Tanvi Rao         |
|Meera Pandey      |
|Tanvi Patel       |
|Rahul Jain        |
|Kavya Bo

## Invalid numerical values


In [64]:
df.select(
    sum(when(col("quantity").cast("double").isNull(), 1).otherwise(0)).alias("invalid_quantity"),
    sum(when(col("unit_price").cast("double").isNull(), 1).otherwise(0)).alias("invalid_unit_price"),
    sum(when(col("discount_percent").cast("double").isNull(), 1).otherwise(0)).alias("invalid_discount")
).show()

+----------------+------------------+----------------+
|invalid_quantity|invalid_unit_price|invalid_discount|
+----------------+------------------+----------------+
|               0|                 0|               0|
+----------------+------------------+----------------+



## Invalid dates

In [65]:
parsed_date = coalesce(
    expr("try_to_timestamp(order_date,'yyyy-MM-dd')"),
    expr("try_to_timestamp(order_date,'dd/MM/yyyy')"),
    expr("try_to_timestamp(order_date,'MM/dd/yyyy')"),
    expr("try_to_timestamp(order_date,'dd-MM-yyyy')"),
    expr("try_to_timestamp(order_date,'MM-dd-yyyy')")
)

df.withColumn("parsed_date", parsed_date).filter(
    col("parsed_date").isNull()
).select("order_date").distinct().show(50, False)

+----------+
|order_date|
+----------+
|2025/07/14|
|31/02/2026|
|not-a-date|
|2025-02-30|
|2026-13-05|
+----------+



## Removing exact duplicate rows

In [66]:
df_silver = df.dropDuplicates()
print("Rows after removing duplicates:", df_silver.count())

Rows after removing duplicates: 990


## Trimming text fields and handle blank strings

In [67]:
for c in df_silver.columns:
    if isinstance(df_silver.schema[c].dataType, StringType):
        df_silver = df_silver.withColumn(
            c,
            when(trim(col(c)) == "", None).otherwise(trim(col(c)))
        )

## Standardize text capitalization


In [68]:
text_columns = ["customer_name", "product", "category", "payment_method", "city", "state", "order_status"]

for c in text_columns:
    df_silver = df_silver.withColumn(c, initcap(col(c)))

## Handling missing text values

In [69]:
df_silver = df_silver.fillna({
    "customer_name": "Unknown",
    "product": "Unknown",
    "category": "Unknown",
    "payment_method": "Unknown",
    "city": "Unknown",
    "state": "Unknown",
    "order_status": "Unknown"
})

df_silver.show(5)

+--------+-----------+------------------+--------------------+---------+--------+----------+----------------+----------------+----------+-----------+----------+------------+
|order_id|customer_id|     customer_name|             product| category|quantity|unit_price|discount_percent|  payment_method|      city|      state|order_date|order_status|
+--------+-----------+------------------+--------------------+---------+--------+----------+----------------+----------------+----------+-----------+----------+------------+
|  100331|      C0005|      Sakshi Joshi|            Football|   Sports|       2|       670|              30|     Credit Card| Bengaluru|  Karnataka|2025-06-06|   Cancelled|
|  100333|      C0263|Siddharth Kulkarni|    Programming Book|    Books|       4|       670|               0|     Net Banking|      Pune|Maharashtra|2025-05-08|   Delivered|
|  100506|      C0228|     Diya Tripathi|Competitive Exam ...|    Books|       6|       610|              12|             Upi|Coim

## Converting numerical columns to double


In [70]:
df_silver = df_silver.withColumn("quantity", col("quantity").cast("double"))
df_silver = df_silver.withColumn("unit_price", col("unit_price").cast("double"))
df_silver = df_silver.withColumn("discount_percent", col("discount_percent").cast("double"))

df_silver.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = false)
 |-- product: string (nullable = false)
 |-- category: string (nullable = false)
 |-- quantity: double (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- discount_percent: double (nullable = true)
 |-- payment_method: string (nullable = false)
 |-- city: string (nullable = false)
 |-- state: string (nullable = false)
 |-- order_date: string (nullable = true)
 |-- order_status: string (nullable = false)



## Handling invalid numerical ranges

In [71]:
df_silver = df_silver.withColumn(
    "quantity",
    when(col("quantity") <= 0, None).otherwise(col("quantity"))
)

df_silver = df_silver.withColumn(
    "unit_price",
    when(col("unit_price") <= 0, None).otherwise(col("unit_price"))
)

df_silver = df_silver.withColumn(
    "discount_percent",
    when((col("discount_percent") < 0) | (col("discount_percent") > 100), None).otherwise(col("discount_percent"))
)

## Imputing missing numerical values using the median

In [72]:
for c in ["quantity", "unit_price", "discount_percent"]:
    median = df_silver.approxQuantile(c, [0.5], 0.01)[0]
    df_silver = df_silver.fillna({c: median})

df_silver.select("quantity", "unit_price", "discount_percent").show(10)

+--------+----------+----------------+
|quantity|unit_price|discount_percent|
+--------+----------+----------------+
|     2.0|     670.0|            30.0|
|     4.0|     670.0|             0.0|
|     6.0|     610.0|            12.0|
|    10.0|   37710.0|            30.0|
|     4.0|    6820.0|            35.0|
|     2.0|     840.0|            35.0|
|     4.0|     770.0|            12.0|
|     3.0|    1140.0|            35.0|
|     3.0|   16610.0|             0.0|
|     2.0|    2170.0|            35.0|
+--------+----------+----------------+
only showing top 10 rows


## Standardizing the date format

In [73]:
parsed_date = coalesce(
    expr("try_to_timestamp(order_date,'yyyy-MM-dd')"),
    expr("try_to_timestamp(order_date,'dd/MM/yyyy')"),
    expr("try_to_timestamp(order_date,'MM/dd/yyyy')"),
    expr("try_to_timestamp(order_date,'dd-MM-yyyy')"),
    expr("try_to_timestamp(order_date,'MM-dd-yyyy')")
)

df_silver = df_silver.withColumn(
    "order_date",
    date_format(parsed_date, "yyyy-MM-dd")
)

df_silver.select("order_date").show(10)

+----------+
|order_date|
+----------+
|2025-06-06|
|2025-05-08|
|2025-06-02|
|2025-12-25|
|2026-06-04|
|2026-03-24|
|2026-04-30|
|2026-07-27|
|2025-05-15|
|2026-08-23|
+----------+
only showing top 10 rows


## Comparing row counts

In [48]:
print("Before Preprocessing:", df.count())
print("After Preprocessing:", df_silver.count())
print("Records Removed:", df.count() - df_silver.count())

Before Preprocessing: 1000
After Preprocessing: 990
Records Removed: 10


In [49]:
df_silver.write.mode("overwrite").parquet("data_lake/silver/preprocessed_sales")

## Creating the Silver csv for GitHub submission

In [50]:
os.makedirs("submission/silver/", exist_ok=True)

temp_path = "submission/silver/temp_sales"

df_silver.coalesce(1).write.mode("overwrite").option("header", True).csv(temp_path)

csv_file = [f for f in os.listdir(temp_path) if f.endswith(".csv")][0]

shutil.copy(
    os.path.join(temp_path, csv_file),
    "submission/silver/sales_silver.csv"
)

shutil.rmtree(temp_path)

In [51]:
silver_check = spark.read.parquet("data_lake/silver/preprocessed_sales")

silver_check.show(5)
silver_check.printSchema()

print("Silver Records:", silver_check.count())

+--------+-----------+------------------+--------------------+---------+--------+----------+----------------+----------------+----------+-----------+----------+------------+
|order_id|customer_id|     customer_name|             product| category|quantity|unit_price|discount_percent|  payment_method|      city|      state|order_date|order_status|
+--------+-----------+------------------+--------------------+---------+--------+----------+----------------+----------------+----------+-----------+----------+------------+
|  100331|      C0005|      Sakshi Joshi|            Football|   Sports|     2.0|     670.0|            30.0|     Credit Card| Bengaluru|  Karnataka|2025-06-06|   Cancelled|
|  100333|      C0263|Siddharth Kulkarni|    Programming Book|    Books|     4.0|     670.0|             0.0|     Net Banking|      Pune|Maharashtra|2025-05-08|   Delivered|
|  100506|      C0228|     Diya Tripathi|Competitive Exam ...|    Books|     6.0|     610.0|            12.0|             Upi|Coim

## Verifying the Silver CSV

In [52]:
silver_csv = spark.read.csv("submission/silver/sales_silver.csv", header=True, inferSchema=True)

silver_csv.show(5)
print("CSV Records:", silver_csv.count())

+--------+-----------+------------------+--------------------+---------+--------+----------+----------------+----------------+----------+-----------+----------+------------+
|order_id|customer_id|     customer_name|             product| category|quantity|unit_price|discount_percent|  payment_method|      city|      state|order_date|order_status|
+--------+-----------+------------------+--------------------+---------+--------+----------+----------------+----------------+----------+-----------+----------+------------+
|  100331|      C0005|      Sakshi Joshi|            Football|   Sports|     2.0|     670.0|            30.0|     Credit Card| Bengaluru|  Karnataka|2025-06-06|   Cancelled|
|  100333|      C0263|Siddharth Kulkarni|    Programming Book|    Books|     4.0|     670.0|             0.0|     Net Banking|      Pune|Maharashtra|2025-05-08|   Delivered|
|  100506|      C0228|     Diya Tripathi|Competitive Exam ...|    Books|     6.0|     610.0|            12.0|             Upi|Coim

In [53]:
from google.colab import files

files.download("submission/silver/sales_silver.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# DATA PREPROCESSING SUMMARY

1. Dataset downloaded from the class GitHub repository.
2. Original dataset stored in the Bronze layer.
3. Duplicate records removed.
4. Missing and blank values handled.
5. Text columns trimmed and standardized.
6. Invalid numerical values handled using median imputation.
7. Dates standardized to `yyyy-MM-dd` format.
8. Processed dataset saved in the Silver layer.
9. Silver data read back and verified.